In [3]:
import pandas as pd
import numpy as np

# Load historical rainfall data
df = pd.read_csv("../data/raw/Kerala-Rainfall-Historical.csv")

# Load documented flood-year evidence
flood_evidence = pd.read_csv("../data/raw/flood_year_evidence.csv")

# Extract documented flood years
flood_years = (
    flood_evidence.loc[
        flood_evidence["FLOOD_LABEL"] == 1, "YEAR"
    ]
    .astype(int)
    .tolist()
)

# Verify that every documented year exists in our dataset
available_years = set(df["YEAR"].astype(int))
matched_years = sorted(set(flood_years) & available_years)
unmatched_years = sorted(set(flood_years) - available_years)

print("Documented flood years:", sorted(flood_years))
print("Matched flood years:", matched_years)
print("Unmatched flood years:", unmatched_years)

# Create a separate labelled dataset
df_labeled = df.copy()

df_labeled["FLOOD_LABEL"] = (
    df_labeled["YEAR"]
    .astype(int)
    .isin(matched_years)
    .astype(int)
)


# DATA QUALITY AND LABEL VALIDATION

# 1. Verify the original dataset
assert df.shape == (117, 19), (
    "Unexpected dataset dimensions."
)

assert df["YEAR"].is_unique, (
    "Duplicate years detected."
)

assert sorted(df["YEAR"].astype(int).tolist()) == list(
    range(1901, 2018)
), "Unexpected or missing years detected."

assert df["SUBDIVISION"].nunique() == 1
assert df["SUBDIVISION"].iloc[0] == "Kerala"

# 2. Verify monthly rainfall columns
monthly_columns = [
    "JAN", "FEB", "MAR", "APR", "MAY", "JUN",
    "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"
]

assert df[monthly_columns].notna().all().all(), (
    "Missing monthly rainfall values detected."
)

assert np.isfinite(
    df[monthly_columns].to_numpy(dtype=float)
).all(), "Non-finite rainfall values detected."

assert (df[monthly_columns] >= 0).all().all(), (
    "Negative rainfall measurements detected."
)

# 3. Verify flood evidence
assert flood_evidence["YEAR"].is_unique, (
    "Duplicate years found in flood evidence."
)

assert set(flood_evidence["FLOOD_LABEL"].unique()) == {1}

assert set(flood_years).issubset(available_years), (
    "Some documented flood years are missing from the dataset."
)

assert matched_years == sorted(flood_years), (
    "Flood-year matching is incomplete."
)

# 4. Verify generated target labels
assert df_labeled["YEAR"].is_unique

assert df_labeled["FLOOD_LABEL"].isin([0, 1]).all()

assert df_labeled["FLOOD_LABEL"].sum() == 6

assert (df_labeled["FLOOD_LABEL"] == 0).sum() == 111

# 5. Verify exact correspondence between years and labels
generated_flood_years = set(
    df_labeled.loc[
        df_labeled["FLOOD_LABEL"] == 1,
        "YEAR"
    ].astype(int)
)

assert generated_flood_years == set(flood_years)

print("All data-quality checks passed!")
print("Total observations:", len(df_labeled))
print("Documented flood years:", len(generated_flood_years))
print("Years not listed:", (df_labeled["FLOOD_LABEL"] == 0).sum())

# Display class distribution
class_counts = df_labeled["FLOOD_LABEL"].value_counts().sort_index()

distribution = pd.DataFrame({
    "Class": ["Non-flood (not listed)", "Documented major flood"],
    "Label": [0, 1],
    "Count": [
        class_counts.get(0, 0),
        class_counts.get(1, 0)
    ]
})

distribution["Percentage"] = (
    distribution["Count"] / len(df_labeled) * 100
).round(2)

display(distribution)

print("\nLabelled dataset shape:", df_labeled.shape)
print("\nSample labelled observations:")
display(
    df_labeled.loc[
        df_labeled["FLOOD_LABEL"] == 1,
        ["YEAR", "ANNUAL", "FLOOD_LABEL"]
    ]
)

Documented flood years: [1924, 1958, 1961, 1992, 2007, 2013]
Matched flood years: [1924, 1958, 1961, 1992, 2007, 2013]
Unmatched flood years: []
All data-quality checks passed!
Total observations: 117
Documented flood years: 6
Years not listed: 111


,Class,Label,Count,Percentage
0,Non-flood (not listed),0,111,94.87
1,Documented major flood,1,6,5.13



Labelled dataset shape: (117, 20)

Sample labelled observations:


,YEAR,ANNUAL,FLOOD_LABEL
23,1924,4226.4,1
57,1958,2923.1,1
60,1961,4257.8,1
91,1992,3239.5,1
106,2007,3489.6,1
112,2013,3255.4,1


In [3]:
from pathlib import Path

# Create processed data directory
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save labelled dataset
output_path = output_dir / "kerala_flood_labeled.csv"

df_labeled.to_csv(output_path, index=False)

print("Labelled dataset saved successfully!")
print("Location:", output_path)
print("Shape:", df_labeled.shape)
print("Flood years:", flood_years)

Labelled dataset saved successfully!
Location: ..\data\processed\kerala_flood_labeled.csv
Shape: (117, 20)
Flood years: [1924, 1958, 1961, 1992, 2007, 2013]


#### Flood-Year Label Definition

The binary target variable FLOOD_LABEL was created using documented major flood years identified in the Kerala State Planning Board's December 2019 committee report (Table 5.7).

A value of 1 represents a year documented as a major flood year in the selected chronology. A value of 0 represents a year not listed in that chronology.

This definition does not imply that every year labelled 0 was completely free of local flooding.

Six positive observations were identified among the 117 years from 1901 to 2017. The 2018 Kerala flood is excluded because it falls outside the dataset period.